In [2]:
# importando bibliotecas
import pandas as pd
import re
import os
import numpy as np


In [3]:
# confirmnando o localhost do min.io
import os
print(os.getenv("MINIO_ENDPOINT"))

None


Comando para reconhecer a pasta na raiz do projeto e conexão com o DuckBD

In [4]:
#os imports somente devem ser adicionados, caso o notebook não reconheça a coinfiguração do pythonpath 
# import os
# import sys

# sys.path.insert(0, os.path.abspath(".."))

from src.utils.database import get_duckdb_connection

con = get_duckdb_connection()


In [5]:
#corre
con.execute("""
SELECT COUNT(*) AS total
FROM read_csv(
    's3://raw/censo_escolar/2025/escola_2025.csv',
    delim=';',
    header=True,
    encoding='latin-1',
    ignore_errors=true,
    strict_mode=false,
    all_varchar=true,
    null_padding=true
);
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total
0,214192


In [8]:
import pandas as pd
import io
import os
from src.ingestion.minio_client import get_s3_client, upload_file

def tratar_planilha_cetic(caminho_raw, pesquisa, ano):
    """
    Função universal para resolver o 'bagunçado' da CETIC (Educação e Domicílios).
    """
    s3 = get_s3_client()
    bucket_raw = "raw"
    bucket_trusted = "trusted"
    
    print(f"📡 Iniciando 'Raio-X' de: {caminho_raw}")
    
    obj = s3.get_object(Bucket=bucket_raw, Key=caminho_raw)
    conteudo = io.BytesIO(obj['Body'].read())
    
    # Lê todas as abas
    abas = pd.read_excel(conteudo, sheet_name=None, header=None)
    
    for nome_aba, df in abas.items():
        # Pula abas de índice ou notas (geralmente com poucas colunas)
        if df.empty or len(df.columns) < 3: continue
        
        try:
            # 1. PEGAR COLUNAS: Na imagem, os nomes estão na linha 3 (index 2)
            # Pegamos da coluna C em diante (index 2 em diante)
            colunas_respostas = df.iloc[2, 2:].dropna().tolist()
            
            # 2. DEFINIR ESTRUTURA: Forçamos o nome das colunas
            # Garantimos que o número de colunas bata com o dataframe
            novas_colunas = ['Categoria', 'Subcategoria'] + colunas_respostas
            df.columns = novas_colunas[:len(df.columns)]
            
            # 3. LIMPAR CABEÇALHO: Os dados começam na linha 4 (index 3 ou 4 dependendo da aba)
            # Vamos detectar onde começa o 'TOTAL' para cortar o lixo de cima
            inicio_dados = df[df['Categoria'].astype(str).str.contains('TOTAL', na=False, case=False)].index[0]
            df_dados = df.iloc[inicio_dados:].reset_index(drop=True)
            
            # 4. RESOLVER CÉLULAS MESCLADAS (O seu maior problema na imagem)
            # O ffill() repete 'REGIÃO' para todas as linhas de Sul, Norte, etc.
            df_dados['Categoria'] = df_dados['Categoria'].ffill()
            
            # 5. TRANSFORMAR PARA FORMATO ANALÍTICO (MELT)
            df_longo = df_dados.melt(
                id_vars=['Categoria', 'Subcategoria'],
                var_name='Equipamento_ou_Segmento',
                value_name='Total'
            )
            
            # Limpeza de números (remove pontos e converte para numérico)
            df_longo['Total'] = pd.to_numeric(
                df_longo['Total'].astype(str).str.replace('.', '').str.replace(',', '.'), 
                errors='coerce'
            )
            
            # Remove linhas vazias ou notas de rodapé que o melt capturou
            df_longo = df_longo.dropna(subset=['Total'])

            # 6. SALVAR NO MINIO (TRUSTED)
            nome_final = f"{nome_aba}_limpo.csv".lower()
            caminho_temp = f"temp_{nome_final}"
            df_longo.to_csv(caminho_temp, index=False, sep=";")
            
            destino_s3 = f"cetic/{pesquisa}/{ano}/{nome_final}"
            upload_file(caminho_temp, bucket_trusted, destino_s3)
            os.remove(caminho_temp)
            
            print(f"✅ Aba {nome_aba} processada com sucesso!")

        except Exception as e:
            print(f"⚠️ Pulei a aba {nome_aba}: Estrutura diferente.")

# No seu main.py, você chamaria assim:
# tratar_planilha_cetic("cetic/domicilios/2025/tic_domicilios_2025.xlsx", "domicilios", "2025")
# tratar_planilha_cetic("cetic/educacao/2024/tic_educacao_2024.xlsx", "educacao", "2024")